# Lab 6b · Sentinel-1 radar — before/after change detection
### Flood & landslide / avalanche mapping when clouds hide the ground

**Objective.** Use Sentinel-1 radar in Google Earth Engine to map **what changed on the ground**
around a monsoon flood/mass-movement event, and judge whether the evidence supports an
**ice–rock avalanche → debris flow** in the source valley.

**Worked example:** the **26 August 2026 Bhote Koshi flood** (central Nepal). It's a *standalone*
exercise — it does **not** depend on Labs 1–6 — and the whole thing is parameterised, so you can
re-point the boxes and dates at any event (including a West Rapti flood).

**Time:** 60–75 min. **Needs:** an authenticated Earth Engine project.

> ⚠️ This is a training exercise on a real disaster in which people died. Treat the output as
> **exploratory remote sensing, not an official damage or cause assessment** — those come from
> NDRRMA, DHM and ICIMOD. All coordinates below are **approximate**: check them on the map in §4
> and move the boxes before trusting any number.

## 0 · Environment setup  *(Colab or local — auto-detected)*

**What this does.** Detects Colab, installs `geemap`/`geedim`, imports Earth Engine, and sets an
output folder. Standalone — nothing from Labs 1–6 is read.

In [ ]:
import os, sys
try:
    import google.colab; IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
    get_ipython().system('pip -q install geemap geedim')
import ee, geemap, numpy as np, pandas as pd
OUT_DIR = '/content/drive/MyDrive/west_rapti/outputs/lab6b' if IN_COLAB else 'outputs/lab6b'
os.makedirs(OUT_DIR, exist_ok=True)
print('running on', 'Colab' if IN_COLAB else 'local', '| outputs ->', OUT_DIR)
import json
_pj = '/content/drive/MyDrive/west_rapti/participant.json' if IN_COLAB else 'participant.json'
PARTICIPANT = json.load(open(_pj)) if os.path.exists(_pj) else {}
EE_PROJECT = PARTICIPANT.get('ee_project', '')
if not EE_PROJECT: print('⚠️  No Earth Engine project set — run 00_Setup.ipynb first (or set EE_PROJECT in CONFIG).')

## 1 · Configuration — **edit only this cell**

Set your Earth Engine project, the event date, the **before/after** windows, and a **control year**
(the same calendar windows one year earlier — it tells you how much change is just normal monsoon
wetness/snowmelt). The three boxes/points are **approximate** — you'll verify them on the map in §4.

In [ ]:
EE_PROJECT = EE_PROJECT or 'your-ee-project-id'                 # <-- your Earth Engine Cloud project
try: ee.Initialize(project=EE_PROJECT)
except Exception: ee.Authenticate(); ee.Initialize(project=EE_PROJECT)

EVENT_DATE          = '2026-08-26'
PRE_START, PRE_END  = '2026-07-10', '2026-08-25'   # BEFORE: several acquisitions, same season
POST_START, POST_END= '2026-08-26', '2026-09-25'   # AFTER
BASELINE_YEAR       = 2025                          # CONTROL: same windows one year earlier

ORBIT_PASS = 'ASCENDING'     # keep ONE pass so geometry is identical pre/post ('ASCENDING' | 'DESCENDING')
POL        = 'VV'            # polarisation for open water ('VV' is darkest over smooth water)

# --- APPROXIMATE geometry (verify & move on the §4 map) : lon0, lat0, lon1, lat1 ---
AOI_BBOX    = [85.80, 28.05, 86.15, 28.40]   # the corridor to study
SOURCE_ZONE = [85.94, 28.20, 86.03, 28.30]   # suspected avalanche start / run-out
PLACES = {'suspected source (Lhende Khola)': [85.98, 28.26],
          'debris-flow corridor (Bhote Koshi)': [85.90, 28.15]}

WATER_THR_DB = -17.0     # VV below this dB ~ open water (tune on the §6 histogram)
CHG_THR_DB   =   3.0     # |Δσ0| above this many dB = a real change hotspot
print('event', EVENT_DATE, '| pre', PRE_START, '->', PRE_END, '| post', POST_START, '->', POST_END,
      '| control', BASELINE_YEAR)

## 2 · How radar change detection works

The flood happened in the monsoon, so optical satellites mostly saw cloud. **Radar sees through
cloud.** Sentinel-1 sends microwave pulses and records the returned energy — the **backscatter**
$\sigma^0$ (in dB), which depends on surface **roughness**, **moisture** and **geometry**:

- fresh **rock / ice blocks**, rough debris → **brighter** ($\sigma^0\uparrow$)
- **stripped vegetation**, smooth **wet mud / fine sediment**, **open water** → **darker** ($\sigma^0\downarrow$)

Compare a **before** stack with an **after** stack, pixel by pixel:

$$\Delta\sigma^0_{dB} \;=\; 10\log_{10}\!\frac{\sigma^0_{post}}{\sigma^0_{pre}} \;=\; \sigma^0_{post,\,dB}-\sigma^0_{pre,\,dB}$$

Large $|\Delta\sigma^0|$ clustered where an avalanche would start and run out — **and absent in a
control year** — is evidence. Three rules keep it honest:

1. **Same orbit & geometry** before and after — otherwise terrain alone fakes change.
2. **Reduce speckle** — radar is grainy; average several dates (in linear power) and smooth lightly.
3. **Check a control year** — the same dates one year earlier show how much change is *normal*.

## 3 · Build the before / after / control radar stacks

**What this does.** Selects Sentinel-1 GRD scenes on **one orbit pass** (same geometry), for the
before, after, and control-year windows. Speckle is reduced by averaging in **linear power** then
converting back to dB, plus a light spatial smooth. `mean_db` returns a VV+VH dB image per window.

In [ ]:
aoi = ee.Geometry.Rectangle(AOI_BBOX)

def s1_coll(d0, d1):
    return (ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(aoi).filterDate(d0, d1)
              .filter(ee.Filter.eq('instrumentMode', 'IW'))
              .filter(ee.Filter.eq('orbitProperties_pass', ORBIT_PASS))
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
              .select(['VV', 'VH']))

def mean_db(coll):
    lin = coll.map(lambda im: ee.Image(10.0).pow(im.divide(10.0)))   # dB -> linear power
    return lin.mean().log10().multiply(10.0).focal_mean(1.5, 'circle', 'pixels').clip(aoi)  # -> dB, despeckled

def shift(d, yr): return pd.Timestamp(d).replace(year=yr).strftime('%Y-%m-%d')

pre   = mean_db(s1_coll(PRE_START,  PRE_END))
post  = mean_db(s1_coll(POST_START, POST_END))
pre_c = mean_db(s1_coll(shift(PRE_START, BASELINE_YEAR),  shift(PRE_END, BASELINE_YEAR)))
post_c= mean_db(s1_coll(shift(POST_START, BASELINE_YEAR), shift(POST_END, BASELINE_YEAR)))
print('scene counts | pre:', s1_coll(PRE_START,PRE_END).size().getInfo(),
      'post:', s1_coll(POST_START,POST_END).size().getInfo(),
      '| control pre:', s1_coll(shift(PRE_START,BASELINE_YEAR),shift(PRE_END,BASELINE_YEAR)).size().getInfo(),
      'post:', s1_coll(shift(POST_START,BASELINE_YEAR),shift(POST_END,BASELINE_YEAR)).size().getInfo())
print('If any count is 0, widen the window or switch ORBIT_PASS.')

## 4 · 🖱️ Verify the study boxes on the map

**What this does.** Draws the *after* VV image with the AOI, the suspected **source zone**, and the
place markers. **This is the step that matters most:** the coordinates in §1 are guesses — pan/zoom
and move `AOI_BBOX` / `SOURCE_ZONE` / `PLACES` until they sit on the real valley, then re-run §3–§8.

In [ ]:
m = geemap.Map(basemap='Esri.WorldImagery'); m.centerObject(aoi, 10)
m.addLayer(post.select(POL), {'min': -25, 'max': 0}, f'AFTER {POL} (dB)')
m.addLayer(pre.select(POL),  {'min': -25, 'max': 0}, f'BEFORE {POL} (dB)', False)
m.addLayer(ee.Image().paint(ee.FeatureCollection([ee.Feature(ee.Geometry.Rectangle(SOURCE_ZONE))]), 0, 2),
           {'palette': 'red'}, 'source zone')
for name, (lo, la) in PLACES.items():
    m.add_marker(location=[la, lo], popup=name)
m

## 5 · The change image  $\Delta\sigma^0_{dB}$

**What this does.** Post minus pre, in dB, for VV and VH. On the diverging palette: **blue = darkening**
(new water, wet mud, stripped vegetation), **red = brightening** (fresh rough debris, blocks). VH is
more sensitive to volume/roughness change; VV to open water.

In [ ]:
dVV = post.select('VV').subtract(pre.select('VV')).rename('dVV')
dVH = post.select('VH').subtract(pre.select('VH')).rename('dVH')
dVV_ctrl = post_c.select('VV').subtract(pre_c.select('VV')).rename('dVV_ctrl')

vis_chg = {'min': -6, 'max': 6, 'palette': ['#2166ac', '#67a9cf', '#f7f7f7', '#ef8a62', '#b2182b']}
m5 = geemap.Map(basemap='Esri.WorldImagery'); m5.centerObject(aoi, 10)
m5.addLayer(dVV, vis_chg, 'ΔVV event (dB)')
m5.addLayer(dVH, vis_chg, 'ΔVH event (dB)', False)
m5.addLayer(dVV_ctrl, vis_chg, 'ΔVV control year (dB)', False)
m5.add_colorbar(vis_chg, label='Δσ⁰ (dB):  blue = darker/wetter,  red = brighter/rougher')
m5

## 6 · Flood / open-water mapping

**What this does.** Open water is a mirror to radar → very low VV. Flag pixels that are **water after
but not before**, and remove **permanent** water (JRC Global Surface Water). Reports the flooded area.
Tune `WATER_THR_DB` on the histogram if the mask over/under-shoots.

In [ ]:
gsw  = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence')
perm = gsw.gt(50).unmask(0)                                  # >50% occurrence = permanent water
water_post = post.select(POL).lt(WATER_THR_DB)
water_pre  = pre.select(POL).lt(WATER_THR_DB)
flood = water_post.And(water_pre.Not()).And(perm.Not()).selfMask().rename('flood')

area_km2 = (flood.multiply(ee.Image.pixelArea()).reduceRegion(
              ee.Reducer.sum(), aoi, 30, maxPixels=1e12).get('flood'))
print('new-water (flood) area:  %.2f km2  (threshold %.0f dB)'
      % ((ee.Number(area_km2).getInfo() or 0)/1e6, WATER_THR_DB))

m6 = geemap.Map(basemap='Esri.WorldImagery'); m6.centerObject(aoi, 10)
m6.addLayer(post.select(POL), {'min': -25, 'max': 0}, f'AFTER {POL}', True, 0.7)
m6.addLayer(perm.selfMask(), {'palette': '#4575b4'}, 'permanent water', False)
m6.addLayer(flood, {'palette': '#d73027'}, 'new water (flood)')
m6

## 7 · Change hotspots — debris vs stripped/wet ground

**What this does.** Splits the change into **brightening** (Δσ⁰ > +`CHG_THR_DB`: fresh rough debris,
blocks) and **darkening** (Δσ⁰ < −`CHG_THR_DB`: new water, wet mud, stripped vegetation). Together
these outline an avalanche start zone and its run-out.

In [ ]:
brighten = dVV.gt(CHG_THR_DB).selfMask()      # rougher: debris / blocks
darken   = dVV.lt(-CHG_THR_DB).selfMask()     # smoother/wetter: water, mud, stripped veg
m7 = geemap.Map(basemap='Esri.WorldImagery'); m7.centerObject(aoi, 11)
m7.addLayer(post.select(POL), {'min': -25, 'max': 0}, 'AFTER VV', True, 0.6)
m7.addLayer(brighten, {'palette': '#b2182b'}, f'brightening > +{CHG_THR_DB:.0f} dB (debris)')
m7.addLayer(darken,   {'palette': '#2166ac'}, f'darkening  < -{CHG_THR_DB:.0f} dB (water/stripped)')
m7.addLayer(ee.Image().paint(ee.FeatureCollection([ee.Feature(ee.Geometry.Rectangle(SOURCE_ZONE))]), 0, 2),
            {'palette': 'yellow'}, 'source zone')
m7

## 8 · Is the change *abnormal*? — event year vs control year

**What this does.** Real event change should appear in the event year but **not** in the control year.
`abnormal` = strong change this year where the control year was quiet. If the hotspots survive this
test and cluster in the source zone / corridor, the radar evidence is consistent with a real
mass-movement + flood — if they vanish, the 'change' was just normal monsoon variability.

In [ ]:
strong_event   = dVV.abs().gt(CHG_THR_DB)
quiet_control  = dVV_ctrl.abs().lt(CHG_THR_DB)
abnormal = strong_event.And(quiet_control).selfMask().rename('abnormal')

a_evt = ee.Number(strong_event.multiply(ee.Image.pixelArea()).reduceRegion(
          ee.Reducer.sum(), aoi, 30, maxPixels=1e12).get('dVV')).getInfo() or 0
a_abn = ee.Number(abnormal.multiply(ee.Image.pixelArea()).reduceRegion(
          ee.Reducer.sum(), aoi, 30, maxPixels=1e12).get('abnormal')).getInfo() or 0
print('strong change (event yr): %.2f km2 | of which abnormal (quiet in %d): %.2f km2 (%.0f%%)'
      % (a_evt/1e6, BASELINE_YEAR, a_abn/1e6, 100*a_abn/max(a_evt,1)))

m8 = geemap.Map(basemap='Esri.WorldImagery'); m8.centerObject(aoi, 11)
m8.addLayer(post.select(POL), {'min': -25, 'max': 0}, 'AFTER VV', True, 0.6)
m8.addLayer(abnormal, {'palette': '#54278f'}, 'abnormal change (event, not control)')
m8

## 9 · Export the maps & wrap up

**What this does.** Writes the change image, the flood mask and the abnormal-change mask to Drive as
batch exports (no timeout). Load them in QGIS / a report.

In [ ]:
def export(img, name):
    t = ee.batch.Export.image.toDrive(image=img.toFloat(), description=name, folder='west_rapti',
                                       region=aoi, scale=20, crs='EPSG:4326', maxPixels=1e12)
    t.start(); print('export started:', name, '(task', t.id, ')')
export(dVV.addBands(dVH), 'lab6b_change_dB')
export(flood.unmask(0),   'lab6b_flood_mask')
export(abnormal.unmask(0),'lab6b_abnormal_change')
print('check the Tasks tab / Drive/west_rapti for the GeoTIFFs.')

### What this shows — and what it can't

- **Floods:** Sentinel-1 GRD amplitude change detection is a standard, robust operational method
  (e.g. Twele et al. 2016). Open-water flood extent at ~10–20 m is reliable; urban flooding is hard.
- **Debris / avalanche run-out:** amplitude change (roughness) is *indicative*, not proof.
- **What would make it conclusive:** **InSAR coherence loss** from Sentinel-1 **SLC** data (processed
  in SNAP or ASF HyP3 — not available inside Earth Engine), plus official NDRRMA / DHM / ICIMOD
  findings and field verification.
- **Snow/ice caveat:** wet snow also darkens radar (Nagler & Rott 2000); avalanche debris is rougher,
  so brighter than surrounding snow (Eckerstorfer & Malnes 2015). Separate these before concluding.

*Method sources: Twele et al. (2016), flood change detection; Nagler & Rott (2000), wet-snow mapping;
Eckerstorfer & Malnes (2015), Sentinel-1 avalanche debris detection.*